# EXP-01: Horizontal resolution (x4)

**Hypothesis.** Reducing the horizontal downsampling of the CRNN from x8 to x4 improves the recognition of small symbols that sit next to other symbols (naturals, flats, slurs, grace notes). Each CTC frame then covers less of the image, so fewer symbols compete for the same frame. The mechanism is a hypothesis: the EXP-00 error analysis only shows that these symbols are the main source of omissions.

**Single change vs EXP-00.** The third max-pool goes from `(2, 2)` to `(2, 1)` and the logit length from `width // 8` to `width // 4`. The parameter count is unchanged (1,896,407). Everything else is identical: data, splits, seed, bucketed pipeline, optimizer, schedule, 8 epochs, fixed 2,000-sample validation subset and best-model criterion.

**Success criteria** (full validation set, measured in the evaluation notebook):
- CER at least 10% lower than EXP-00 (0.0067 -> 0.0060 or less).
- Higher recall for `accidental.natural` (EXP-00: 0.768) and for slurs, and fewer omissions.
- Secondary metric: CER ignoring a trailing `barline-L1`, which neutralizes the final-barline label inconsistency found in EXP-00.

**Reference (EXP-00).** Subset CER 0.0073 (best epoch 8), full-validation CER 0.0067, train-subset CER 0.0057, 3.94 h of training.

**Limitation.** One run per experiment; run-to-run variation is not measured, hence the 10% threshold.

**After a disconnect:** `Runtime -> Run all`. Section 7 resumes automatically from the last checkpoint in `artifacts/checkpoints/exp01/`.

## 1. Setup

### 1.1. Imports

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import islice
import json, math, os, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

### 1.2. Drive mount, paths and extraction

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/NoteVision')
RAW_DIR = PROJECT_DIR / 'data_raw'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
LOCAL_DIR = Path('/content/primus')
ARCHIVE = RAW_DIR / 'primus.tgz'
EXPECTED_SAMPLES = 87_678
SEED = 42

def count_samples(root):
    n = 0
    for pkg in ('package_aa', 'package_ab'):
        p = root / pkg
        if p.exists():
            n += sum(1 for d in p.iterdir() if d.is_dir() and not d.name.startswith('._'))
    return n

if count_samples(LOCAL_DIR) != EXPECTED_SAMPLES:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $LOCAL_DIR

n = count_samples(LOCAL_DIR)
print('Samples on disk:', n)
assert n == EXPECTED_SAMPLES, 'Incomplete extraction'

Mounted at /content/drive
Samples on disk: 87678


### 1.3. Preprocessing functions and metrics

In [3]:
IMG_HEIGHT = 128

def load_vocab(vocab_csv):
    """Return (tokens, token_to_id, n_tokens)"""
    df = pd.read_csv(vocab_csv, dtype=str, keep_default_na=False)
    tokens = df['token'].tolist()
    return tokens, {t: i for i, t in enumerate(tokens)}, len(tokens)

def encode_labels(df, token_to_id, max_len):
    """Dense int32 array [N, max_len] (zero padded) and true lengths [N]."""
    labels = np.zeros((len(df), max_len), dtype=np.int32)
    lengths = np.zeros(len(df), dtype=np.int32)
    for i, seq in enumerate(df['label']):
        ids = [token_to_id[t] for t in seq.split()]
        labels[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return labels, lengths

def decode_ids(ids, tokens):
    return ' '.join(tokens[i] for i in ids)

def image_path(root, package, sid):
    return Path(root) / package / sid / f'{sid}.png'

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_png(raw, channels=3) # palette (mode P) -> RGB
    gray = tf.image.rgb_to_grayscale(img) # uint8 [H, W, 1]
    h = tf.cast(tf.shape(gray)[0], tf.float32)
    w = tf.cast(tf.shape(gray)[1], tf.float32)
    new_w = tf.maximum(1, tf.cast(tf.round(w * IMG_HEIGHT / h), tf.int32))
    gray = tf.image.resize(tf.cast(gray, tf.float32), (IMG_HEIGHT, new_w),
                           method='bilinear', antialias=True)
    gray = tf.clip_by_value(gray, 0.0, 255.0)
    return 1.0 - gray / 255.0, new_w # ink = 1, background/padding = 0

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (ref[i-1] != hyp[j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + (ref[i-1] != hyp[j-1]):
            ops.append(('hit' if ref[i-1] == hyp[j-1] else 'sub', ref[i-1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(('del', ref[i-1]))
            i -= 1
        else:
            ops.append(('ins', None))
            j -= 1
    return d[n][m], ops

def evaluate_sequences(refs, hyps, families):
    tot_ref = tot_err = tot_err_shape = exact = 0
    per_fam = defaultdict(Counter)
    for r, h in zip(refs, hyps):
        dist, ops = align(r, h)
        tot_err += dist
        tot_ref += len(r)
        exact += int(r == h)
        tot_err_shape += align([families[i] for i in r], [families[i] for i in h])[0]
        for op, tok in ops:
            if tok is not None:
                per_fam[families[tok]][op] += 1
    metrics = {'cer': tot_err / tot_ref, 'seq_acc': exact / len(refs),
               'cer_shape': tot_err_shape / tot_ref,
               'n_sequences': len(refs), 'n_tokens': tot_ref}
    rows = []
    for fam, c in per_fam.items():
        support = c['hit'] + c['sub'] + c['del']
        rows.append({'family': fam, 'support': support, 'recall': c['hit'] / support})
    fam_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)
    return metrics, fam_df

# Sanity checks
_fam = ['a', 'a', 'b']
assert align([0, 1, 2], [0, 1, 2])[0] == 0
assert align([0, 1, 2], [0, 2])[0] == 1
assert align([0, 1, 2], [0, 1, 1])[0] == 1
_m, _ = evaluate_sequences([[0, 1, 2]], [[0, 0, 2]], _fam)
assert abs(_m['cer'] - 1/3) < 1e-9 and _m['cer_shape'] == 0.0 and _m['seq_acc'] == 0.0
print('Metric sanity checks passed.')

Metric sanity checks passed.


### 1.4. Bootstrap

In [4]:
parts = {s: pd.read_csv(ARTIFACTS_DIR / f'split_{s}.csv', dtype=str)
         for s in ('train', 'val', 'test')}
for d in parts.values():
    d['seq_len'] = d['seq_len'].astype(int)

tokens, token_to_id, _ = load_vocab(ARTIFACTS_DIR / 'vocab_agnostic.csv')
cfg = json.loads((ARTIFACTS_DIR / 'preprocessing_config.json').read_text())
summ = json.loads((ARTIFACTS_DIR / 'split_summary.json').read_text())

# Guard against drift between notebooks
assert IMG_HEIGHT == cfg['img_height'] and len(tokens) == cfg['vocab_size']
assert {s: len(d) for s, d in parts.items()} == {s: summ[f'n_{s}'] for s in parts}
MAX_LABEL_LEN = cfg['max_label_len']
print('Bootstrap OK:', {s: len(d) for s, d in parts.items()})

Bootstrap OK: {'train': 70138, 'val': 8772, 'test': 8768}


## 2. GPU and Seed

In [5]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(SEED)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Width-Bucketed Pipeline

In [6]:
# Boundaries every 100 px up to 2200, then coarser. Each batch is padded to (boundary - 1),
# so the whole training run sees only ~25 distinct input shapes.
BOUNDS = list(range(600, 2201, 100)) + [2600, 3000, 3600]
MAX_PIXELS = 48000 # batch_size * padded_width budget (~36 images at 1266 px, ~5 GB peak)
MAX_BATCH = 64
BATCH_SIZES = [max(4, min(MAX_BATCH, MAX_PIXELS // b)) for b in BOUNDS] + [4] # last bucket (>= 3600) stays empty
print('Buckets:', len(BOUNDS) + 1, '| batch size per bucket:', BATCH_SIZES)

# Safety: the widest resized image must fit under the top boundary
sz = pd.read_csv(ARTIFACTS_DIR / 'image_sizes.csv')
max_w = int(np.round(sz.width * IMG_HEIGHT / sz.height).max())
assert max_w < BOUNDS[-1], f'widest image ({max_w}) does not fit in the buckets'


def build_dataset_bucketed(df, root, token_to_id, max_label_len, training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width, 'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.bucket_by_sequence_length(
        element_length_func=lambda e: e['width'],
        bucket_boundaries=BOUNDS, bucket_batch_sizes=BATCH_SIZES,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)},
        pad_to_bucket_boundary=True,
        drop_remainder=training) # fixed batch shape in training; keep every sample in val/test
    return ds.prefetch(tf.data.AUTOTUNE)

Buckets: 21 | batch size per bucket: [64, 64, 60, 53, 48, 43, 40, 36, 34, 32, 30, 28, 26, 25, 24, 22, 21, 18, 16, 13, 4]


## 4. CRNN Model (x4)

`MODEL_CFG` holds the architecture choices. `reduction=8` reproduces EXP-00 exactly (same layers in the same order); `reduction=4` is EXP-01. `build_model` is used by the smoke test below and by the training loop in 6.3.

In [7]:
CTC_BLANK = 0 # class 0 = blank; token id i -> class i + 1
NUM_CLASSES = len(tokens) + 1 # 758 tokens + CTC blank
families = [t.split('-')[0] for t in tokens]

MODEL_CFG = {'lstm_units': 128, 'dropout': 0.25, 'reduction': 4}

# Horizontal reduction -> pooling layout (height is always reduced x16)
POOLS = {8: ((2, 2), (2, 2), (2, 2), (2, 1)),  # EXP-00: width /2 three times
         4: ((2, 2), (2, 2), (2, 1), (2, 1))}  # EXP-01: width /2 twice

class CRNN(keras.Model):
    def __init__(self, num_classes, lstm_units=128, dropout=0.25, reduction=8, **kw):
        super().__init__(**kw)
        assert reduction in POOLS, f'unsupported reduction: {reduction}'
        self.reduction = reduction
        filters = (32, 64, 128, 256)
        self.convs = [layers.Conv2D(f, 3, padding='same', use_bias=False) for f in filters]
        self.bns = [layers.BatchNormalization() for _ in filters]
        self.pools = [layers.MaxPool2D(p) for p in POOLS[reduction]]
        self.proj = layers.Dense(256, activation='relu')
        self.drop = layers.Dropout(dropout)
        self.rnn1 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.rnn2 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))
        self.classifier = layers.Dense(num_classes)

    def call(self, inputs, training=False):
        x, widths = inputs['image'], inputs['width']
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = pool(tf.nn.relu(bn(conv(x), training=training)))
        x = tf.transpose(x, [0, 2, 1, 3])
        s = tf.shape(x)
        x = tf.reshape(x, [s[0], s[1], x.shape[2] * x.shape[3]])
        x = self.drop(self.proj(x), training=training)
        logit_len = widths // self.reduction
        mask = tf.sequence_mask(logit_len, maxlen=s[1]) # ignore padding in the BiLSTM
        x = self.rnn1(x, mask=mask, training=training)
        x = self.rnn2(self.drop(x, training=training), mask=mask, training=training)
        return self.classifier(self.drop(x, training=training)), logit_len

def build_model(cfg):
    model = CRNN(NUM_CLASSES, lstm_units=cfg['lstm_units'], dropout=cfg['dropout'],
                 reduction=cfg['reduction'])
    model({'image': tf.zeros([1, IMG_HEIGHT, 256, 1]), 'width': tf.constant([256])})  # build weights
    return model

### 4.1. Shape smoke test

In [8]:
tf.keras.utils.set_random_seed(SEED)
smoke = build_model(MODEL_CFG)
R = MODEL_CFG['reduction']

# 1) Shape logic on a synthetic input (padded width 600; real widths 599 and 300)
out, ln = smoke({'image': tf.zeros([2, IMG_HEIGHT, 600, 1]), 'width': tf.constant([599, 300])},
                training=False)
assert tuple(out.shape) == (2, 600 // R, NUM_CLASSES) and ln.numpy().tolist() == [599 // R, 300 // R]

# 2) One real bucketed training batch
b = next(iter(build_dataset_bucketed(parts['train'], LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                                     training=True, seed=SEED)))
logits, logit_len = smoke({'image': b['image'], 'width': b['width']}, training=False)
print('image:', tuple(b['image'].shape), '| logits:', tuple(logits.shape))
assert logits.shape[1] == b['image'].shape[2] // R
assert logits.shape[-1] == NUM_CLASSES
assert bool(tf.reduce_all(logit_len <= logits.shape[1]))
assert bool(tf.reduce_all(logit_len >= b['label_length'])) # CTC needs T >= label length

n_params = sum(int(np.prod(v.shape)) for v in smoke.trainable_variables)
print('Trainable params:', f'{n_params:,}')
assert n_params == 1_896_407, 'parameter count differs from EXP-00'
del smoke

image: (40, 128, 1199, 1) | logits: (40, 299, 759)
Trainable params: 1,896,407


## 5. CTC Loss, Training Step and Greedy Decoding

In [9]:
def ctc_loss_fn(labels, label_len, logits, logit_len):
    loss = tf.nn.ctc_loss(labels=labels + 1, # token id i -> class i + 1
                          logits=tf.cast(logits, tf.float32),
                          label_length=label_len, logit_length=logit_len,
                          logits_time_major=False, blank_index=CTC_BLANK)
    return tf.reduce_mean(loss)

def make_steps(model, optimizer):
    """Build tf.functions bound to THIS model and optimizer (call again for each new model)."""
    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32),
        tf.TensorSpec([None, MAX_LABEL_LEN], tf.int32),
        tf.TensorSpec([None], tf.int32)])
    def train_step(image, width, label, label_len):
        with tf.GradientTape() as tape:
            logits, logit_len = model({'image': image, 'width': width}, training=True)
            loss = ctc_loss_fn(label, label_len, logits, logit_len)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    @tf.function(input_signature=[
        tf.TensorSpec([None, IMG_HEIGHT, None, 1], tf.float32),
        tf.TensorSpec([None], tf.int32)])
    def infer(image, width):
        return model({'image': image, 'width': width}, training=False)

    return train_step, infer

def greedy_decode(logits, lengths, blank=CTC_BLANK):
    """argmax per step -> collapse repeats -> drop blanks -> back to token ids 0..757."""
    ids = np.argmax(logits, axis=-1)
    out = []
    for seq, n in zip(ids, lengths):
        seq = seq[:n]
        changed = np.concatenate([[True], seq[1:] != seq[:-1]])
        seq = seq[changed]
        out.append((seq[seq != blank] - 1).tolist())
    return out

def predict(batches):
    refs, hyps = [], []
    for b in batches:
        logits, logit_len = infer(b['image'], b['width'])
        hyps += greedy_decode(logits.numpy(), logit_len.numpy())
        lab, ln = b['label'].numpy(), b['label_length'].numpy()
        refs += [lab[i, :ln[i]].tolist() for i in range(len(ln))]
    return refs, hyps

# Decode sanity check (no model): blank=0, token id i = class i + 1
def _fake(classes):
    z = np.zeros((1, len(classes), NUM_CLASSES), dtype=np.float32)
    for t, c in enumerate(classes):
        z[0, t, c] = 1.0
    return z

assert greedy_decode(_fake([0, 3, 3, 0, 5, 5, 0]), [7]) == [[2, 4]]
assert greedy_decode(_fake([3, 0, 3]), [3]) == [[2, 2]] # repeat separated by blank survives
print('Decode sanity checks passed.')

Decode sanity checks passed.


## 6. Training Setup

### 6.1. Config, schedule and steps per epoch

In [10]:
CKPT_DIR = ARTIFACTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
RUN_DIR = CKPT_DIR / 'exp01'

class WarmupCosine(keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup, then cosine decay to lr_final; a pure function of the step."""
    def __init__(self, peak, final, warmup, total):
        super().__init__()
        self.peak, self.final = float(peak), float(final)
        self.warmup, self.total = int(warmup), int(total)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak * (step + 1.0) / self.warmup
        prog = tf.clip_by_value((step - self.warmup) / float(max(1, self.total - self.warmup)), 0.0, 1.0)
        cos = self.final + 0.5 * (self.peak - self.final) * (1.0 + tf.cos(math.pi * prog))
        return tf.where(step < self.warmup, warm, cos)

    def get_config(self):
        return {'peak': self.peak, 'final': self.final, 'warmup': self.warmup, 'total': self.total}

def compute_steps_per_epoch(df):
    """Exact steps per epoch of the bucketed train pipeline (drop_remainder), from image headers only."""
    cache = ARTIFACTS_DIR / 'train_steps_per_epoch.json'
    sig = {'bounds': BOUNDS, 'batch_sizes': BATCH_SIZES, 'n_train': len(df)}
    if cache.exists():
        c = json.loads(cache.read_text())
        if c['sig'] == sig:
            return c['steps_per_epoch']
    widths = []
    for p, i in zip(df['package'], df['id']):
        with Image.open(image_path(LOCAL_DIR, p, i)) as im:
            w, h = im.size
        widths.append(max(1, int(np.round(np.float32(w) * IMG_HEIGHT / np.float32(h)))))
    bucket = np.searchsorted(BOUNDS, np.array(widths), side='right')
    counts = np.bincount(bucket, minlength=len(BATCH_SIZES))
    steps = int(sum(int(n) // bs for n, bs in zip(counts, BATCH_SIZES)))
    cache.write_text(json.dumps({'sig': sig, 'steps_per_epoch': steps, 'bucket_counts': counts.tolist()}))
    return steps

STEPS_PER_EPOCH = compute_steps_per_epoch(parts['train'])

CFG = {
    'exp_id': 'EXP-01', 'seed': SEED, 'epochs': 8, 'steps_per_epoch': STEPS_PER_EPOCH,
    'lr_peak': 1e-3, 'lr_final': 1e-5, 'warmup_steps': 200, 'clipnorm': 5.0,
    **MODEL_CFG, 'max_pixels': MAX_PIXELS, 'max_batch': MAX_BATCH,
    'val_subset_n': 2000, 'early_stop_from_epoch': 4, 'patience': 3,
}

# Fixed validation subset, shared by all experiments
val_sub = parts['val'].sample(CFG['val_subset_n'], random_state=SEED)
ids_path = ARTIFACTS_DIR / 'val_subset_ids.csv'
if ids_path.exists():
    saved = pd.read_csv(ids_path, dtype=str)
    assert saved['id'].tolist() == val_sub['id'].tolist(), 'val subset changed'
else:
    val_sub[['package', 'id']].to_csv(ids_path, index=False)

total = CFG['epochs'] * STEPS_PER_EPOCH
print(f'steps/epoch: {STEPS_PER_EPOCH} | total steps: {total} | '
      f'est. {total * 1.5 / 3600:.1f}-{total * 2.0 / 3600:.1f} h at 1.5-2.0 s/step '
      f'(rough estimate for x4; EXP-00 took 3.94 h at ~1.0 s/step)')
print('Config:', CFG)

steps/epoch: 1830 | total steps: 14640 | est. 6.1-8.1 h at 1.5-2.0 s/step (rough estimate for x4; EXP-00 took 3.94 h at ~1.0 s/step)
Config: {'exp_id': 'EXP-01', 'seed': 42, 'epochs': 8, 'steps_per_epoch': 1830, 'lr_peak': 0.001, 'lr_final': 1e-05, 'warmup_steps': 200, 'clipnorm': 5.0, 'lstm_units': 128, 'dropout': 0.25, 'reduction': 4, 'max_pixels': 48000, 'max_batch': 64, 'val_subset_n': 2000, 'early_stop_from_epoch': 4, 'patience': 3}


### 6.2. Checkpoint helpers

In [11]:
def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1))
    os.replace(tmp, path)

def save_ckpt(run_dir, model, optimizer, state):
    """Write to the alternate slot; the pointer (latest.json) is updated last."""
    run_dir.mkdir(parents=True, exist_ok=True)
    state['n_ckpts'] += 1
    slot = run_dir / f"slot{state['n_ckpts'] % 2}"
    slot.mkdir(exist_ok=True)
    tmp_w = slot / 'tmp.weights.h5'
    model.save_weights(tmp_w)
    os.replace(tmp_w, slot / 'model.weights.h5')
    tmp_o = slot / 'tmp_opt.npz'
    with open(tmp_o, 'wb') as f:
        np.savez(f, **{f'v{i}': v.numpy() for i, v in enumerate(optimizer.variables)})
    os.replace(tmp_o, slot / 'optimizer.npz')
    write_json_atomic(slot / 'state.json', state)
    if hasattr(os, 'sync'):
        os.sync()
    write_json_atomic(run_dir / 'latest.json',
                      {'slot': slot.name, 'global_step': state['global_step'], 'epoch': state['epoch']})

def load_ckpt(run_dir, model, optimizer):
    ptr = run_dir / 'latest.json'
    if not ptr.exists():
        return None
    slot = run_dir / json.loads(ptr.read_text())['slot']
    state = json.loads((slot / 'state.json').read_text())
    model.load_weights(slot / 'model.weights.h5')
    data = np.load(slot / 'optimizer.npz')
    opt_vars = optimizer.variables
    assert len(data.files) == len(opt_vars), 'optimizer variable count mismatch'
    for i, v in enumerate(opt_vars):
        arr = data[f'v{i}']
        assert tuple(v.shape) == arr.shape, f'optimizer variable {i}: shape mismatch'
        v.assign(arr)
    assert int(optimizer.iterations.numpy()) == state['global_step'], 'optimizer.iterations != global_step'
    return state

def save_best(run_dir, model, entry):
    tmp = run_dir / 'tmp_best.weights.h5'
    model.save_weights(tmp)
    os.replace(tmp, run_dir / 'best.weights.h5')
    write_json_atomic(run_dir / 'best.json', entry)

### 6.3. Training loop with resume

In [12]:
def new_state(cfg):
    return {'config': dict(cfg), 'global_step': 0, 'epoch': 1, 'step_in_epoch': 0,
            'epoch_loss_sum': 0.0, 'epoch_loss_n': 0, 'epoch_train_sec': 0.0, 'train_sec_total': 0.0,
            'resume_count': 0, 'n_ckpts': 0, 'best_cer': float('inf'), 'best_epoch': 0,
            'bad_epochs': 0, 'history': [], 'finished': False, 'stop_reason': None}

def run_training(cfg, run_dir, train_df, val_df, ckpt_every=400, max_steps_this_call=None,
                 session_limit_min=None, log_every=200):
    """Fresh start or resume from run_dir. max_steps_this_call simulates a crash (no save)."""
    global infer  # predict() uses the global `infer`; rebind it to this model
    tf.keras.utils.set_random_seed(cfg['seed'])
    model = build_model(cfg)
    total = cfg['epochs'] * cfg['steps_per_epoch']
    sched = WarmupCosine(cfg['lr_peak'], cfg['lr_final'], cfg['warmup_steps'], total)
    opt = keras.optimizers.Adam(sched, clipnorm=cfg['clipnorm'])
    opt.build(model.trainable_variables)

    state = load_ckpt(run_dir, model, opt)
    if state is None:
        state = new_state(cfg)
        print('Starting fresh.')
    else:
        assert state['config'] == dict(cfg), f"config differs from checkpoint: {state['config']}"
        state['resume_count'] += 1
        print(f"Resumed: epoch {state['epoch']}, step_in_epoch {state['step_in_epoch']}, "
              f"global_step {state['global_step']}, lr {float(sched(state['global_step'])):.2e}")
    if state['finished']:
        print('Already finished:', state['stop_reason'])
        return model, opt, state

    train_step, infer = make_steps(model, opt)
    val_ds = build_dataset_bucketed(val_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN)
    t_session, steps_this_call = time.time(), 0

    while state['epoch'] <= cfg['epochs'] and not state['finished']:
        ep = state['epoch']
        ds = build_dataset_bucketed(train_df, LOCAL_DIR, token_to_id, MAX_LABEL_LEN, training=True,
                                    seed=cfg['seed'] + 1000 * ep + state['resume_count'])
        remaining = cfg['steps_per_epoch'] - state['step_in_epoch']
        t_last = time.time()
        for b in islice(ds, remaining):
            loss = float(train_step(b['image'], b['width'], b['label'], b['label_length']))
            if not math.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at step {state['global_step']}; last checkpoint kept")
            now = time.time()
            state['epoch_train_sec'] += now - t_last
            state['train_sec_total'] += now - t_last
            state['global_step'] += 1
            state['step_in_epoch'] += 1
            state['epoch_loss_sum'] += loss
            state['epoch_loss_n'] += 1
            steps_this_call += 1
            if state['global_step'] % log_every == 0:
                print(f"  ep {ep} step {state['step_in_epoch']}/{cfg['steps_per_epoch']} "
                      f"(global {state['global_step']}) | loss {loss:8.2f} | "
                      f"lr {float(sched(state['global_step'])):.2e}")
            if state['global_step'] % ckpt_every == 0:
                save_ckpt(run_dir, model, opt, state)
            if max_steps_this_call and steps_this_call >= max_steps_this_call:
                print(f"SIMULATED INTERRUPT at global step {state['global_step']} (nothing saved now)")
                return model, opt, state
            if session_limit_min and (time.time() - t_session) / 60 >= session_limit_min:
                save_ckpt(run_dir, model, opt, state)
                print('Session time limit reached: checkpoint saved, stopping cleanly.')
                return model, opt, state
            t_last = time.time()  # excludes checkpoint time from the training clock

        # ---- end of epoch: validate on the fixed subset ----
        refs, hyps = predict(val_ds)
        m, _ = evaluate_sequences(refs, hyps, families)
        entry = {'epoch': ep, 'global_step': state['global_step'],
                 'train_loss': state['epoch_loss_sum'] / max(1, state['epoch_loss_n']),
                 'val_cer': m['cer'], 'val_seq_acc': m['seq_acc'], 'val_cer_shape': m['cer_shape'],
                 'lr_end': float(sched(state['global_step'])),
                 'epoch_train_min': state['epoch_train_sec'] / 60}
        entry = {k: float(v) for k, v in entry.items()}
        state['history'].append(entry)
        if m['cer'] < state['best_cer']:
            state['best_cer'], state['best_epoch'], state['bad_epochs'] = float(m['cer']), ep, 0
            save_best(run_dir, model, entry)
            tag = ' *best*'
        else:
            state['bad_epochs'] += 1
            tag = f" (no improvement x{state['bad_epochs']})"
        print(f"EPOCH {ep} | loss {entry['train_loss']:.2f} | val CER {m['cer']:.4f} | "
              f"seq_acc {m['seq_acc']:.3f} | cer_shape {m['cer_shape']:.4f} | "
              f"{entry['epoch_train_min']:.1f} min{tag}")
        write_json_atomic(run_dir / 'history.json', state['history'])

        state['epoch'] += 1
        state['step_in_epoch'] = 0
        state['epoch_loss_sum'], state['epoch_loss_n'], state['epoch_train_sec'] = 0.0, 0, 0.0
        if ep >= cfg['early_stop_from_epoch'] and state['bad_epochs'] >= cfg['patience']:
            state['finished'], state['stop_reason'] = True, 'early_stop'
        elif state['epoch'] > cfg['epochs']:
            state['finished'], state['stop_reason'] = True, 'epochs_done'
        save_ckpt(run_dir, model, opt, state)

    print('Finished:', state['stop_reason'])
    return model, opt, state

## 7. Launch EXP-01

Logs print every 200 steps. Things to check in the first minutes:
- **Speed:** EXP-00 needed about 3.2 min per 200 steps. Expected to take at maximum 7 min per 200 steps (over ~8.5 h in total).
- **Epoch 1:** the validation CER should be on the order of 0.03 (EXP-00: 0.0317).

In [13]:
model, opt, state = run_training(CFG, RUN_DIR, parts['train'], val_sub,
                                 ckpt_every=400, session_limit_min=None)

Resumed: epoch 7, step_in_epoch 620, global_step 11600, lr 1.14e-04
  ep 7 step 820/1830 (global 11800) | loss     0.58 | lr 1.02e-04
  ep 7 step 1020/1830 (global 12000) | loss     0.75 | lr 8.94e-05
  ep 7 step 1220/1830 (global 12200) | loss     0.50 | lr 7.81e-05
  ep 7 step 1420/1830 (global 12400) | loss     0.73 | lr 6.76e-05
  ep 7 step 1620/1830 (global 12600) | loss     1.78 | lr 5.80e-05
  ep 7 step 1820/1830 (global 12800) | loss     0.45 | lr 4.91e-05
EPOCH 7 | loss 0.77 | val CER 0.0075 | seq_acc 0.864 | cer_shape 0.0063 | 49.5 min *best*
  ep 8 step 190/1830 (global 13000) | loss     1.10 | lr 4.12e-05
  ep 8 step 390/1830 (global 13200) | loss     0.73 | lr 3.41e-05
  ep 8 step 590/1830 (global 13400) | loss     0.37 | lr 2.79e-05
  ep 8 step 790/1830 (global 13600) | loss     0.75 | lr 2.26e-05
  ep 8 step 990/1830 (global 13800) | loss     0.34 | lr 1.82e-05
  ep 8 step 1190/1830 (global 14000) | loss     0.22 | lr 1.48e-05
  ep 8 step 1390/1830 (global 14200) | loss 

## 8. Results

### 8.1. Training curves vs EXP-00 (fixed 2,000-sample validation subset)

In [14]:
h00 = pd.DataFrame(json.loads((CKPT_DIR / 'exp00' / 'history.json').read_text()))
h01 = pd.DataFrame(json.loads((RUN_DIR / 'history.json').read_text()))
cols = ['epoch', 'val_cer', 'val_seq_acc', 'val_cer_shape', 'epoch_train_min']
cmp = h00[cols].merge(h01[cols], on='epoch', how='right', suffixes=('_exp00', '_exp01'))
display(cmp.round(4))

,epoch,val_cer_exp00,val_seq_acc_exp00,val_cer_shape_exp00,epoch_train_min_exp00,val_cer_exp01,val_seq_acc_exp01,val_cer_shape_exp01,epoch_train_min_exp01
0,1.0,0.0317,0.5920,0.0257,29.4756,0.0504,0.4050,0.0411,48.6261
1,2.0,0.0156,0.7605,0.0125,29.4351,0.0150,0.7585,0.0133,46.7119
2,3.0,0.0107,0.8150,0.0089,29.4188,0.0111,0.8105,0.0093,46.6385
3,4.0,0.0089,0.8450,0.0074,29.7121,0.0098,0.8305,0.0082,46.6637
4,5.0,0.0082,0.8530,0.0070,29.6914,0.0088,0.8445,0.0074,46.9454
5,6.0,0.0077,0.8605,0.0062,29.6023,0.0080,0.8555,0.0067,46.9480
6,7.0,0.0075,0.8655,0.0061,29.5250,0.0075,0.8640,0.0063,49.4673
7,8.0,0.0073,0.8695,0.0060,29.5260,0.0073,0.8640,0.0062,47.6437


### 8.2. Row for experiments/results.csv

In [15]:
h = pd.DataFrame(state['history'])
b = h.loc[h['val_cer'].idxmin()]
row = {'exp_id': 'EXP-01', 'date': time.strftime('%Y-%m-%d'), 'epochs_run': len(h),
       'best_epoch': int(b['epoch']), 'lr_schedule': 'warmup 200 + cosine 1e-3 to 1e-5',
       'batch_policy': 'bucketed; 48000 px; max 64', 'val_subset_cer': round(b['val_cer'], 4),
       'seq_acc': round(b['val_seq_acc'], 4), 'cer_shape': round(b['val_cer_shape'], 4),
       'train_time_h': round(state['train_sec_total'] / 3600, 2),
       'notes': 'Horizontal reduction x4 (pools 2x2, 2x2, 2x1, 2x1); Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4'}
print(pd.DataFrame([row]).to_csv(index=False))

exp_id,date,epochs_run,best_epoch,lr_schedule,batch_policy,val_subset_cer,seq_acc,cer_shape,train_time_h,notes
EXP-01,2026-11-04,8,8,warmup 200 + cosine 1e-3 to 1e-5,bucketed; 48000 px; max 64,0.0073,0.864,0.0062,6.33,"Horizontal reduction x4 (pools 2x2, 2x2, 2x1, 2x1); Adam clipnorm 5; val subset 2000; early stop patience 3 from epoch 4"



## 9. Next step

Evaluate `best.weights.h5` of `checkpoints/exp01/` on the full validation set in `04_evaluation.ipynb`: set `EXP_ID` and `RUN_DIR`, and use this notebook's `CRNN` (with `reduction`), reading `reduction` from the experiment config. The test set is evaluated once, in the final comparison.